# PF00042: complete protein-set-to-graph pipeline

This notebook completes the required pipeline before data exploration:

```text
protein FASTA -> Biopython / BLAST / DEDAL pair scores -> quality record -> separate graphs
```

The alignment method, graph threshold rule, graph threshold, and correlation-check thresholds are explicit configuration values. The example graph thresholds below are only software smoke-test settings. They are not scientific choices and will be decided after exploration in notebook 03.

In [1]:
import subprocess
import sys
from pathlib import Path

import networkx as nx
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_DIRECTORY = PROJECT_ROOT / "data/processed/pfam_pf00042_globin_pilot"
FASTA_PATH = DATASET_DIRECTORY / "PF00042_globin_pilot.fasta"
METADATA_PATH = DATASET_DIRECTORY / "PF00042_globin_pilot_metadata.tsv"
SCORE_DIRECTORY = PROJECT_ROOT / "outputs/tables/pfam_pf00042_correlation"
GRAPH_DIRECTORY = PROJECT_ROOT / "outputs/graphs/pfam_pf00042_pipeline"

## 1. Pipeline configuration

Each algorithm produces its own graph. `top_n=13` retains the strongest 13 available pairs for each method, giving equal edge counts for a simple pipeline check. Change the rule to `absolute` or `percentile` as required later.

Correlation thresholds intentionally remain `None`: the pipeline records correlations and overlap but does not label them good or poor until notebook 03 provides evidence for suitable cutoffs.

In [2]:
RUN_SCORING = True
METHODS_TO_RUN = ("biopython", "blast", "dedal")

GRAPH_SPECS = {
    "biopython": {"rule": "top_n", "threshold": 13},
    "blast": {"rule": "top_n", "threshold": 13},
    "dedal": {"rule": "top_n", "threshold": 13},
}

CORRELATION_CHECKS = {
    "minimum_rho": None,
    "minimum_overlap_fraction": None,
    "minimum_pairs": None,
}

## 2. Load the protein set

The FASTA supplies the protein identifiers and sequences. The metadata table supplies node attributes such as subgroup, organism, taxonomy, and domain length.

In [3]:
metadata = pd.read_csv(METADATA_PATH, sep="\t")
protein_ids = metadata["uniprot_accession"].tolist()
expected_pair_count = len(protein_ids) * (len(protein_ids) - 1) // 2
print(f"Proteins: {len(protein_ids)}")
print(f"Unique pairs: {expected_pair_count}")
display(metadata[["uniprot_accession", "organism", "subgroup", "domain_length"]])

Proteins: 12
Unique pairs: 66


,uniprot_accession,organism,subgroup,domain_length
0,P02204,Cyprinus carpio (Common carp),myoglobin,114
1,P02200,Alligator mississippiensis (American alligator),myoglobin,117
2,P02206,Heterodontus portusjacksoni (Port Jackson shark),myoglobin,116
3,P02143,Heterodontus portusjacksoni (Port Jackson shark),beta-like haemoglobin,112
4,P02133,Xenopus laevis (African clawed frog),beta-like haemoglobin,117
5,P04443,Mus musculus (Mouse),beta-like haemoglobin,117
6,P02020,Lepidosiren paradoxus (South American lungfish),alpha-like haemoglobin,113
7,P06714,Equus caballus (Horse),alpha-like haemoglobin,111
8,P01967,Bos grunniens (Wild yak) (Bos mutus grunniens),alpha-like haemoglobin,111
9,P09187,Medicago sativa (Alfalfa),divergent globin,116


## 3. Produce or reuse method scores

The scoring runner saves one canonical row for every unique protein pair. Existing outputs are reused, so DEDAL is not recalculated unnecessarily. On a new protein set, including `dedal` will run its slower local model.

In [4]:
if RUN_SCORING:
    scoring_command = [
        sys.executable,
        str(PROJECT_ROOT / "scripts/run_pf00042_correlation.py"),
        "--fasta",
        str(FASTA_PATH),
        "--output-directory",
        str(SCORE_DIRECTORY),
        "--methods",
        *METHODS_TO_RUN,
        "--threads",
        "4",
    ]
    subprocess.run(scoring_command, cwd=PROJECT_ROOT, check=True)

paired_scores = pd.read_csv(SCORE_DIRECTORY / "paired_scores.tsv", sep="\t")
assert len(paired_scores) == expected_pair_count
display(paired_scores.head())

Reusing biopython: /Users/davin/Desktop/Protein Seq Alignment/outputs/tables/pfam_pf00042_correlation/biopython_pairs.tsv
Reusing blast: /Users/davin/Desktop/Protein Seq Alignment/outputs/tables/pfam_pf00042_correlation/blast_pairs.tsv
Reusing dedal: /Users/davin/Desktop/Protein Seq Alignment/outputs/tables/pfam_pf00042_correlation/dedal_pairs.tsv
Merged 66 canonical pairs
Completed methods: biopython, blast, dedal
Manifest: /Users/davin/Desktop/Protein Seq Alignment/outputs/tables/pfam_pf00042_correlation/run_manifest.json


,protein_a,protein_b,biopython_score,blast_raw_score,blast_bit_score,blast_evalue,blast_percent_identity,blast_alignment_length,blast_query_coverage_hsp,dedal_sw_score,...,dedal_gap_count,dedal_alignment_length,biopython_score_rank,biopython_score_percentile,blast_bit_score_rank,blast_bit_score_percentile,dedal_sw_score_rank,dedal_sw_score_percentile,dedal_homology_logit_rank,dedal_homology_logit_percentile
0,P01967,P02020,248.0,257.0,103.0,4.410000e-34,44.248,113.0,100.0,39.242275,...,2,113,3.0,0.969697,3.0,0.96,3.0,0.969697,3.0,0.969697
1,P01967,P02133,205.0,218.0,88.6,4.210000e-28,40.000,110.0,94.0,33.116547,...,6,117,10.0,0.863636,10.0,0.82,6.0,0.924242,6.0,0.924242
2,P01967,P02143,167.0,179.0,73.6,2.980000e-22,38.889,108.0,96.0,28.826246,...,1,112,16.0,0.772727,16.0,0.70,13.0,0.818182,13.0,0.818182
3,P01967,P02200,81.0,NaN,NaN,NaN,NaN,NaN,NaN,16.491179,...,6,117,30.0,0.560606,NaN,NaN,29.0,0.575758,29.0,0.575758
4,P01967,P02204,103.0,97.0,42.0,5.040000e-10,27.193,114.0,96.0,15.432405,...,7,116,27.0,0.606061,27.0,0.48,31.0,0.545455,31.0,0.545455


## 4. Build separate algorithm graphs

The graph runner applies each method's own configured rule and writes separate GraphML, edge-table, node-table, summary, and provenance files. Missing BLAST hits remain unavailable and can never become zero-score edges.

In [5]:
graph_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts/build_method_graphs.py"),
    "--fasta",
    str(FASTA_PATH),
    "--pairs",
    str(SCORE_DIRECTORY / "paired_scores.tsv"),
    "--metadata",
    str(METADATA_PATH),
    "--correlations",
    str(SCORE_DIRECTORY / "spearman_correlations.tsv"),
    "--output-directory",
    str(GRAPH_DIRECTORY),
]
for method, specification in GRAPH_SPECS.items():
    graph_command.extend(
        [
            "--graph",
            f"{method}:{specification['rule']}:{specification['threshold']}",
        ]
    )
for option, value in CORRELATION_CHECKS.items():
    if value is not None:
        graph_command.extend([f"--{option.replace('_', '-')}", str(value)])
subprocess.run(graph_command, cwd=PROJECT_ROOT, check=True)

Built 3 separate graph(s) from 12 proteins
Quality thresholds configured: False
Manifest: /Users/davin/Desktop/Protein Seq Alignment/outputs/graphs/pfam_pf00042_pipeline/graph_run_manifest.json


CompletedProcess(args=['/Users/davin/Desktop/Protein Seq Alignment/.venv/bin/python', '/Users/davin/Desktop/Protein Seq Alignment/scripts/build_method_graphs.py', '--fasta', '/Users/davin/Desktop/Protein Seq Alignment/data/processed/pfam_pf00042_globin_pilot/PF00042_globin_pilot.fasta', '--pairs', '/Users/davin/Desktop/Protein Seq Alignment/outputs/tables/pfam_pf00042_correlation/paired_scores.tsv', '--metadata', '/Users/davin/Desktop/Protein Seq Alignment/data/processed/pfam_pf00042_globin_pilot/PF00042_globin_pilot_metadata.tsv', '--correlations', '/Users/davin/Desktop/Protein Seq Alignment/outputs/tables/pfam_pf00042_correlation/spearman_correlations.tsv', '--output-directory', '/Users/davin/Desktop/Protein Seq Alignment/outputs/graphs/pfam_pf00042_pipeline', '--graph', 'biopython:top_n:13', '--graph', 'blast:top_n:13', '--graph', 'dedal:top_n:13'], returncode=0)

## 5. Verify graph outputs

The smoke-test configuration should create three different graph objects with the complete protein set. The quality table records evidence but remains `not_evaluated` until explicit correlation thresholds are configured.

In [6]:
summaries = pd.read_csv(GRAPH_DIRECTORY / "graph_summaries.tsv", sep="\t")
quality = pd.read_csv(GRAPH_DIRECTORY / "quality_flags.tsv", sep="\t")
graphs = {}
for method, specification in GRAPH_SPECS.items():
    label = str(specification["threshold"]).replace(".", "p")
    path = GRAPH_DIRECTORY / f"{method}_{specification['rule']}_{label}.graphml"
    graphs[method] = nx.read_graphml(path)
    assert set(graphs[method]) == set(protein_ids)

display(summaries)
display(quality)

,method,threshold_rule,threshold,nodes,edges,isolates,connected_components,density,average_clustering
0,biopython,top_n,13.0,12,13,3,5,0.19697,0.583333
1,blast,top_n,13.0,12,13,3,5,0.19697,0.583333
2,dedal,top_n,13.0,12,13,4,6,0.19697,0.450000


,score_a,score_b,spearman_rho,n_pairs,overlap_fraction,minimum_rho,minimum_overlap_fraction,minimum_pairs,status,flags
0,biopython_score,blast_bit_score,0.993036,50,0.757576,NaN,NaN,NaN,not_evaluated,NaN
1,biopython_score,dedal_sw_score,0.873681,66,1.000000,NaN,NaN,NaN,not_evaluated,NaN
2,blast_bit_score,dedal_sw_score,0.891749,50,0.757576,NaN,NaN,NaN,not_evaluated,NaN


## Pipeline status

The pipeline now accepts a protein set, runs selectable alignment methods, records configurable correlation checks, and emits a separate saved graph for each method under explicit threshold settings. Notebook 03 is the next stage: explore the data and results, then replace these smoke-test graph settings with justified thresholds and sensitivity ranges.